# Эксперимент по diarization спикеров

Сравнение модели Hugging Face:
- `pyannote/speaker-diarization-3.1`

Подвыборка: **0.002%** от датасета `data/calls-3-quarantine.jsonl`.

## 2) Импорты и конфиг

In [22]:
import os
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchaudio
import torchcodec
from pyannote.audio import Pipeline

# Проверка CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно GPU: {torch.cuda.device_count()}")
else:
    print("CUDA не найдена, будет использоваться CPU")

Используется устройство: cuda
GPU: NVIDIA GeForce RTX 4060
Доступно GPU: 1


In [23]:
PROJECT_DIR = Path.cwd()
JSONL_PATH = PROJECT_DIR / "data" / "calls-3-quarantine.jsonl"

OUT_DIR = PROJECT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUT_DIR / "diarization_experiment"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
SAMPLE_PCT = 0.002
MIN_SAMPLE = 1

MODELS = [
    "pyannote/speaker-diarization-3.1",
]

np.random.seed(SEED)

## 3) Загрузка JSONL и подготовка подвыборки 0.002%

In [24]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


df = read_jsonl(JSONL_PATH)
required = {"id", "path", "duration_sec"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"В JSONL отсутствуют поля: {missing}")

df = df.drop_duplicates(subset=["id"]).copy()
df["path"] = df["path"].astype(str)
df["exists"] = df["path"].map(lambda p: Path(p).exists())
df_ok = df[df["exists"]].copy()

if len(df_ok) == 0:
    raise RuntimeError("Нет доступных файлов по полю path. Проверь пути в JSONL.")

sample_n = max(MIN_SAMPLE, int(round(len(df_ok) * SAMPLE_PCT)))
sample_n = min(sample_n, len(df_ok))
sample_df = df_ok.sample(n=sample_n, random_state=SEED).reset_index(drop=True)

print(f"Всего доступных файлов: {len(df_ok)}")
print(f"Выбрано в эксперимент: {sample_n} ({100*sample_n/len(df_ok):.6f}%)")
sample_df.head()

Всего доступных файлов: 51744
Выбрано в эксперимент: 103 (0.199057%)


,id,path,file_name,duration_sec,size_bytes,exists
0,9450df88675f07c12142bffeeed0a9e1bedce4e7,Z:\calls\dataset\2025\12\18\1766039757.126564.wav,1766039757.126564.wav,12.08,193324,True
1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,1766391440.276622.wav,53.52,856364,True
2,54cbc5758f76c97496696f389ff96b27bbfb0ea5,Z:\calls\dataset\2025\12\24\1766572725.435044.wav,1766572725.435044.wav,52.22,835564,True
3,e87980fd1833d05aa36ce256fff4c2bafb48aec2,Z:\calls\dataset\2025\12\30\1767076086.623749.wav,1767076086.623749.wav,14.22,227564,True
4,4e75fddb021f9fa0052409a4abcec6bad2735a44,Z:\calls\dataset\2025\12\22\1766406416.310675.wav,1766406416.310675.wav,11.28,180524,True


## 4) Инициализация Hugging Face токена и моделей

In [25]:
HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
if not HF_TOKEN:
    raise EnvironmentError(
        'HF_TOKEN не найден. В PowerShell выполни: $env:HF_TOKEN="<token>" и перезапусти kernel.'
    )


def load_pipeline_compat(model_id: str, hf_token: str):
    trial_kwargs = [
        {"token": hf_token},
        {"use_auth_token": hf_token},
    ]

    last_error = None
    for kwargs in trial_kwargs:
        try:
            pipeline = Pipeline.from_pretrained(model_id, **kwargs)
            if pipeline is not None:
                pipeline.to(device)
            return pipeline, None
        except TypeError as e:
            # несовместимость сигнатуры между версиями pyannote
            last_error = e
            continue
        except Exception as e:
            return None, str(e)

    return None, f"Не удалось подобрать аргумент авторизации: {last_error}"


loaded = {}
for model_id in MODELS:
    pipeline, err = load_pipeline_compat(model_id, HF_TOKEN)
    loaded[model_id] = {"pipeline": pipeline, "load_error": err}
    if err:
        print(f"[LOAD FAIL] {model_id}: {str(err)[:220]}")
    else:
        print(f"[LOAD OK] {model_id}")

[LOAD OK] pyannote/speaker-diarization-3.1


## 5) Функции инференса (единый формат результатов)

In [26]:
def annotation_to_segments(annotation):
    segments = []
    for segment, _, speaker in annotation.itertracks(yield_label=True):
        start = float(segment.start)
        end = float(segment.end)
        if end > start:
            segments.append(
                {
                    "start": start,
                    "end": end,
                    "speaker": str(speaker),
                    "duration": end - start,
                }
            )
    segments.sort(key=lambda x: x["start"])
    return segments


def run_diarization(
    pipe,
    audio_path: str,
    audio_duration_sec: float,
    min_speakers: int = 1,
    max_speakers: int = 2,
):
    t0 = time.perf_counter()

    waveform, sample_rate = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    waveform = waveform.to(device)

    audio_in_memory = {
        "waveform": waveform,
        "sample_rate": sample_rate,
        "uri": audio_path,
    }

    annotation = pipe(
        audio_in_memory, min_speakers=min_speakers, max_speakers=max_speakers
    )

    # Поддержка pyannote.audio >= 4.0.0, где возвращается DiarizeOutput
    if hasattr(annotation, "speaker_diarization"):
        annotation = annotation.speaker_diarization

    runtime_sec = time.perf_counter() - t0

    segments = annotation_to_segments(annotation)
    n_speakers = len(set(s["speaker"] for s in segments)) if segments else 0
    speech_total = sum(s["duration"] for s in segments) if segments else 0.0

    rtf = runtime_sec / max(audio_duration_sec, 1e-9)
    coverage = min(1.0, speech_total / max(audio_duration_sec, 1e-9))

    return {
        "runtime_sec": runtime_sec,
        "rtf": rtf,
        "segments_count": len(segments),
        "n_speakers_pred": n_speakers,
        "speech_coverage": coverage,
        "segments": segments,
    }

## 6) Запуск эксперимента на подвыборке

In [27]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = RESULTS_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

rows = []
segments_rows = []

for model_id in MODELS:
    pipe = loaded[model_id]["pipeline"]
    load_error = loaded[model_id]["load_error"]

    for _, r in tqdm(sample_df.iterrows(), total=len(sample_df), desc=model_id):
        file_id = r["id"]
        audio_path = r["path"]
        duration_sec = float(r["duration_sec"])

        if pipe is None:
            rows.append(
                {
                    "model": model_id,
                    "id": file_id,
                    "path": audio_path,
                    "duration_sec": duration_sec,
                    "status": "load_failed",
                    "error": load_error,
                }
            )
            continue

        try:
            result = run_diarization(pipe, audio_path, duration_sec)
            rows.append(
                {
                    "model": model_id,
                    "id": file_id,
                    "path": audio_path,
                    "duration_sec": duration_sec,
                    "status": "ok",
                    "error": "",
                    "runtime_sec": result["runtime_sec"],
                    "rtf": result["rtf"],
                    "segments_count": result["segments_count"],
                    "n_speakers_pred": result["n_speakers_pred"],
                    "speech_coverage": result["speech_coverage"],
                }
            )

            for seg in result["segments"]:
                segments_rows.append(
                    {"model": model_id, "id": file_id, "path": audio_path, **seg}
                )

        except Exception as e:
            rows.append(
                {
                    "model": model_id,
                    "id": file_id,
                    "path": audio_path,
                    "duration_sec": duration_sec,
                    "status": "inference_failed",
                    "error": str(e)[:1000],
                }
            )

results_df = pd.DataFrame(rows)
segments_df = pd.DataFrame(segments_rows)

display(results_df.head())

pyannote/speaker-diarization-3.1:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\corey\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)
c:\Users\corey\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\corey\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


,model,id,path,duration_sec,status,error,runtime_sec,rtf,segments_count,n_speakers_pred,speech_coverage
0,pyannote/speaker-diarization-3.1,9450df88675f07c12142bffeeed0a9e1bedce4e7,Z:\calls\dataset\2025\12\18\1766039757.126564.wav,12.08,ok,,0.426423,0.035300,10,2,0.504294
1,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,53.52,ok,,1.284680,0.024004,22,2,0.659929
2,pyannote/speaker-diarization-3.1,54cbc5758f76c97496696f389ff96b27bbfb0ea5,Z:\calls\dataset\2025\12\24\1766572725.435044.wav,52.22,ok,,1.252561,0.023986,29,2,0.711258
3,pyannote/speaker-diarization-3.1,e87980fd1833d05aa36ce256fff4c2bafb48aec2,Z:\calls\dataset\2025\12\30\1767076086.623749.wav,14.22,ok,,0.187862,0.013211,5,2,1.000000
4,pyannote/speaker-diarization-3.1,4e75fddb021f9fa0052409a4abcec6bad2735a44,Z:\calls\dataset\2025\12\22\1766406416.310675.wav,11.28,ok,,0.106900,0.009477,3,1,0.323138


## 7) Сохранение результатов

In [28]:
sample_df.to_csv(run_dir / "sample_files.csv", index=False, encoding="utf-8-sig")
results_df.to_csv(run_dir / "results_summary.csv", index=False, encoding="utf-8-sig")
segments_df.to_csv(run_dir / "results_segments.csv", index=False, encoding="utf-8-sig")

with (run_dir / "run_config.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "jsonl_path": str(JSONL_PATH),
            "sample_pct": SAMPLE_PCT,
            "seed": SEED,
            "models": MODELS,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Результаты сохранены в:", run_dir)

Результаты сохранены в: d:\project\multicriteria-dialog-audit-ml\data\diarization_experiment\20260301_202341
